In [ ]:

from collections import defaultdict

import pandas as pd
from dotenv import load_dotenv

from fabric_client import Credentials, FabricClient, FabricContainer, scan

# Read FABRIC_CLI_TENANT_ID,FABRIC_CLI_CLIENT_ID and FABRIC_CLI_CLIENT_SECRET
load_dotenv()

creds = Credentials.env()

container = FabricContainer()
container.credentials.override(creds)
client = container.client()
client.set_log_level("INFO")
data = defaultdict(list)

def display(name:str) -> pd.DataFrame:
    """Display the result dataframe."""
    return pd.DataFrame(data[name])

In [ ]:
# fetch data for all kinds of metadata in a nested iteration
filtered = [w for w in await client.workspaces if w.display_name is not None]
async for workspace in scan(filtered):
    data['workspaces'].append(workspace.model_dump())
    async for dataset in workspace.datasets:
        data['datasets'].append(dataset.model_dump())
        async for query in dataset.queries:
            data['queries'].append(query.model_dump())
        async for refresh in dataset.refreshes:
            data['refreshes'].append(refresh.model_dump())

    async for dataflow in workspace.dataflows:
        data["dataflows"].append(dataflow.model_dump())
        async for query in dataflow.queries:
            data["queries"].append(query)
        async for transaction in dataflow.transactions:
            data['transactions'].append(transaction.model_dump())

    async for report in workspace.reports:
        data['reports'].append(report.model_dump())

In [ ]:
display("workspaces")